# SINCON 2026 요모조모 — Kaggle Notebook / Google Colab 재현

최종 제출 `submission_fixcancel_geombawi_0.144388.csv` (Public 0.144388)를 대회 원본에서 다시 만든다. 위에서부터 차례로 실행하면 된다.

- **기본 (CPU, 약 2분)**: 저장된 개별 모델 예측 8개에서 시작해 섞기 → 후처리 → 취소 행사 → 검바위 보정을 다시 계산하고 단계마다 저장본과 비교한다.
- **`RETRAIN_CPU = True` (CPU, 약 10~30분 추가)**: kdbwls 모델(v7·역별 LightGBM)을 다시 학습해 저장본과 비교한다.
- 패키지는 런타임에 설치된 것을 그대로 쓰고, 없는 것만 버전 고정 없이 설치한다 (Python 3.10 + pandas 2.3, Python 3.13 + pandas 3.0 양쪽에서 같은 결과를 확인함). GPU 불필요.
- 대회 원본: Kaggle은 대회 데이터를 노트북에 추가하면 `/kaggle/input/` 아래에서 자동으로 찾는다. Colab은 `train.csv`·`test.csv`·`metaData.csv`·`sample_submission.csv` 4개를 `/content/raw/`에 올리거나 `RAW_DIR`에 폴더를 적는다.
- **GitHub 저장소에는 CSV 파일이 없다.** 제출 ZIP(`SINCON_2026_요모조모.zip`)을 풀어 Kaggle Dataset 또는 Colab `/content/`에 올리고 `REPO`(환경변수 `SINCON_REPO`)에 그 폴더를 적는다. 저장소만 받은 경우에는 `setup_raw_data.py` 단계 이후 개별 모델 예측 CSV가 없어 진행되지 않는다.

In [ ]:
import os, sys, glob, shutil, subprocess, importlib
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IN_KAGGLE else Path("/content") if Path("/content").exists() else Path.cwd()

REPO_URL = "https://github.com/aaeejak/SINCON-2026.git"
REPO = Path(os.environ.get("SINCON_REPO", str(WORK / "SINCON-2026")))   # 저장소 위치 (없으면 git clone)
RAW_DIR = os.environ.get("SINCON_RAW", "")                              # 대회 원본 4개 파일 폴더 (비우면 자동 탐색)
RETRAIN_CPU = os.environ.get("RETRAIN_CPU", "0") == "1"                 # kdbwls 모델 재학습

def sh(cmd, cwd=None):
    print(f"\n$ {cmd}\n  (위치: {cwd or os.getcwd()})", flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, env={**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONDONTWRITEBYTECODE": "1"})
    if r.returncode != 0:
        raise RuntimeError(f"실패 (exit {r.returncode}): {cmd}")

print("Kaggle" if IN_KAGGLE else "Colab/기타", "| 작업 폴더", WORK, "| Python", sys.version.split()[0])

## 1. 저장소 받기

In [ ]:
if not (REPO / "setup_raw_data.py").exists():
    sh(f"git clone --depth 1 {REPO_URL} '{REPO}'")
print(sorted(p.name for p in REPO.iterdir()))

## 2. 패키지 확인 (없는 것만 설치)

In [ ]:
need = {"pandas": "pandas", "numpy": "numpy", "lightgbm": "lightgbm", "scipy": "scipy"}
missing = []
for mod, pkg in need.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    sh(f"{sys.executable} -m pip install -q " + " ".join(missing))
for mod in need:
    print(f"{mod:10s} {importlib.import_module(mod).__version__}")

## 3. 대회 원본 배치 (SHA-256 확인)

In [ ]:
def find_raw():
    if RAW_DIR:
        return Path(RAW_DIR)
    for r in ["/kaggle/input", "/content/raw", "/content/drive/MyDrive", "/content", str(WORK)]:
        for t in sorted(glob.glob(f"{r}/**/test.csv", recursive=True)):
            d = Path(t).parent
            if str(REPO) in str(d):
                continue
            if all((d / f).exists() for f in ("train.csv", "metaData.csv", "sample_submission.csv")):
                return d
    raise FileNotFoundError("대회 원본 4개 파일을 찾지 못했습니다. RAW_DIR에 폴더를 적어 주세요.")

raw = find_raw()
print("원본 폴더:", raw)
sh(f"{sys.executable} setup_raw_data.py '{raw}'", cwd=REPO)

## 4. 최종 제출 재현
개별 모델 예측 → [1] 섞기 원본 → [2] 후처리 → [3] 팀 섞기 → [4] 보정 복원 → [5] 취소 행사 → [6] 검바위 보정. 마지막 줄에 `재현 성공`이 나와야 한다.

In [ ]:
K = REPO / "SINCON_2026_요모조모_kdbwls"
sh(f"{sys.executable} code/run_all.py" + (" --retrain" if RETRAIN_CPU else ""), cwd=K)

## 5. 최종 파일 확인

In [ ]:
import numpy as np, pandas as pd
final, saved = K / "work/final_A.csv", K / "predictions/final_A_fixcancel_geombawi.csv"
a, b = pd.read_csv(final), pd.read_csv(saved)
te = pd.read_csv(K / "data/raw/test.csv", usecols=["example_id"])
assert list(a.columns) == ["example_id", "boardings"] and len(a) == 157194 and a.example_id.is_unique
assert (a.example_id.to_numpy() == te.example_id.to_numpy()).all()
assert np.isfinite(a.boardings).all() and (a.boardings >= 0).all()
print("다시 만든 파일 vs 제출 원본 최대 log 차이:", float(np.abs(np.log1p(a.boardings) - np.log1p(b.boardings)).max()))
shutil.copy(final, WORK / "submission.csv")
print("저장:", WORK / "submission.csv")